In [1]:
import pandas as pd

# ============================================================
# DATASET METADATA / STATE-LEVEL AUDIT
# ============================================================

# 1. Load dataset
input_file = 'E:/Projects/BTP/Data/Raw_data/INDIA/Final_data.csv'

df = pd.read_csv(input_file)

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

# ------------------------------------------------------------
# 2. Create proper date
# ------------------------------------------------------------

df['Date'] = pd.to_datetime(
    df[['year', 'mon', 'day']].rename(columns={'mon': 'month'}),
    errors='coerce'
)

# ------------------------------------------------------------
# 3. Overall dataset metadata
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL METADATA")
print("=" * 70)

print(f"Unique states/UTs: {df['state_ut'].nunique():,}")
print(f"Unique districts: {df['district'].nunique():,}")
print(f"Unique diseases: {df['Disease'].nunique():,}")

print(f"Earliest date: {df['Date'].min()}")
print(f"Latest date:   {df['Date'].max()}")

print(f"Total cases:  {df['Cases'].sum():,.0f}")
print(f"Total deaths: {df['Deaths'].sum():,.0f}")

# ------------------------------------------------------------
# 4. State-level metadata
# ------------------------------------------------------------

state_metadata = (
    df.groupby('state_ut')
    .agg(
        rows=('state_ut', 'size'),
        unique_districts=('district', 'nunique'),
        unique_dates=('Date', 'nunique'),
        first_date=('Date', 'min'),
        last_date=('Date', 'max'),
        total_cases=('Cases', 'sum'),
        total_deaths=('Deaths', 'sum'),
        nonzero_case_records=('Cases', lambda x: (x > 0).sum()),
        zero_case_records=('Cases', lambda x: (x == 0).sum()),
        diseases=('Disease', 'nunique')
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Calculate useful percentages
# ------------------------------------------------------------

state_metadata['nonzero_case_%'] = (
    state_metadata['nonzero_case_records']
    / state_metadata['rows']
    * 100
)

state_metadata['zero_case_%'] = (
    state_metadata['zero_case_records']
    / state_metadata['rows']
    * 100
)

# ------------------------------------------------------------
# 6. Calculate missing values per state
# ------------------------------------------------------------

missing_cases = (
    df.groupby('state_ut')['Cases']
    .apply(lambda x: x.isna().sum())
    .reset_index(name='missing_cases')
)

missing_deaths = (
    df.groupby('state_ut')['Deaths']
    .apply(lambda x: x.isna().sum())
    .reset_index(name='missing_deaths')
)

state_metadata = state_metadata.merge(
    missing_cases,
    on='state_ut'
)

state_metadata = state_metadata.merge(
    missing_deaths,
    on='state_ut'
)

# ------------------------------------------------------------
# 7. Sort states by number of observations
# ------------------------------------------------------------

state_metadata = state_metadata.sort_values(
    by='rows',
    ascending=False
)

# ------------------------------------------------------------
# 8. Print state-level metadata
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STATE / UT LEVEL METADATA")
print("=" * 70)

print(
    state_metadata.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 9. Save metadata
# ------------------------------------------------------------

output_file = (
    'E:/Projects/BTP/Data/Clean/INDIA/'
    'state_level_metadata.csv'
)

state_metadata.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(f"Metadata saved to:")
print(output_file)

print("\nTop 10 states by number of records:")

print(
    state_metadata[
        ['state_ut', 'rows', 'unique_districts',
         'unique_dates', 'total_cases']
    ]
    .head(10)
    .to_string(index=False)
)

DATASET OVERVIEW
Total rows: 8,985
Total columns: 15
Columns: ['Unnamed: 0', 'week_of_outbreak', 'state_ut', 'district', 'Disease', 'Cases', 'Deaths', 'day', 'mon', 'year', 'Latitude', 'Longitude', 'preci', 'LAI', 'Temp']

OVERALL METADATA
Unique states/UTs: 36
Unique districts: 791
Unique diseases: 22
Earliest date: 2009-05-04 00:00:00
Latest date:   2022-06-29 00:00:00


ValueError: Unknown format code 'f' for object of type 'str'

In [2]:
import pandas as pd
import os

# 1. Load dataset
input_file = 'E:/Projects/BTP/Data/Raw_data/INDIA/Final_data.csv'

df = pd.read_csv(input_file)

# Convert Cases and Deaths to numeric
df['Cases'] = pd.to_numeric(df['Cases'], errors='coerce')
df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce')

# Create proper date
df['Date'] = pd.to_datetime(
    df[['year', 'mon', 'day']].rename(columns={'mon': 'month'}),
    errors='coerce'
)

# ------------------------------------------------------------
# 3. Overall dataset metadata
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL METADATA")
print("=" * 70)

print(f"Unique states/UTs: {df['state_ut'].nunique():,}")
print(f"Unique districts: {df['district'].nunique():,}")
print(f"Unique diseases: {df['Disease'].nunique():,}")

print(f"Earliest date: {df['Date'].min()}")
print(f"Latest date:   {df['Date'].max()}")

print(f"Total cases:  {df['Cases'].sum():,.0f}")
print(f"Total deaths: {df['Deaths'].sum():,.0f}")

# ------------------------------------------------------------
# 4. State-level metadata
# ------------------------------------------------------------

state_metadata = (
    df.groupby('state_ut')
    .agg(
        rows=('state_ut', 'size'),
        unique_districts=('district', 'nunique'),
        unique_dates=('Date', 'nunique'),
        first_date=('Date', 'min'),
        last_date=('Date', 'max'),
        total_cases=('Cases', 'sum'),
        total_deaths=('Deaths', 'sum'),
        nonzero_case_records=('Cases', lambda x: (x > 0).sum()),
        zero_case_records=('Cases', lambda x: (x == 0).sum()),
        diseases=('Disease', 'nunique')
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Calculate useful percentages
# ------------------------------------------------------------

state_metadata['nonzero_case_%'] = (
    state_metadata['nonzero_case_records']
    / state_metadata['rows']
    * 100
)

state_metadata['zero_case_%'] = (
    state_metadata['zero_case_records']
    / state_metadata['rows']
    * 100
)

# ------------------------------------------------------------
# 6. Calculate missing values per state
# ------------------------------------------------------------

missing_cases = (
    df.groupby('state_ut')['Cases']
    .apply(lambda x: x.isna().sum())
    .reset_index(name='missing_cases')
)

missing_deaths = (
    df.groupby('state_ut')['Deaths']
    .apply(lambda x: x.isna().sum())
    .reset_index(name='missing_deaths')
)

state_metadata = state_metadata.merge(
    missing_cases,
    on='state_ut'
)

state_metadata = state_metadata.merge(
    missing_deaths,
    on='state_ut'
)

# ------------------------------------------------------------
# 7. Sort states by number of observations
# ------------------------------------------------------------

state_metadata = state_metadata.sort_values(
    by='rows',
    ascending=False
)

# ------------------------------------------------------------
# 8. Print state-level metadata
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STATE / UT LEVEL METADATA")
print("=" * 70)

print(
    state_metadata.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 9. Save metadata
# ------------------------------------------------------------

output_file = (
    'E:/Projects/BTP/Data/Clean/INDIA/'
    'state_level_metadata.csv'
)

state_metadata.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(f"Metadata saved to:")
print(output_file)

print("\nTop 10 states by number of records:")

print(
    state_metadata[
        ['state_ut', 'rows', 'unique_districts',
         'unique_dates', 'total_cases']
    ]
    .head(10)
    .to_string(index=False)
)


OVERALL METADATA
Unique states/UTs: 36
Unique districts: 791
Unique diseases: 22
Earliest date: 2009-05-04 00:00:00
Latest date:   2022-06-29 00:00:00
Total cases:  796,427
Total deaths: 4,047

STATE / UT LEVEL METADATA
                   state_ut  rows  unique_districts  unique_dates first_date  last_date  total_cases  total_deaths  nonzero_case_records  zero_case_records  diseases  nonzero_case_%  zero_case_%  missing_cases  missing_deaths
                Maharashtra  1195                38           985 2009-07-23 2022-06-15      53575.0         340.0                  1195                  0        14       100.00000          0.0              0             851
                  Karnataka  1097                57           888 2009-06-09 2022-06-14      39532.0         125.0                  1097                  0        11       100.00000          0.0              0             867
                West Bengal   889                42           596 2009-05-05 2022-05-08     178830.0 

In [3]:
# ============================================================
# DISEASE-LEVEL METADATA
# ============================================================

disease_metadata = (
    df.groupby('Disease')
    .agg(
        rows=('Disease', 'size'),
        states=('state_ut', 'nunique'),
        districts=('district', 'nunique'),
        unique_dates=('Date', 'nunique'),
        first_date=('Date', 'min'),
        last_date=('Date', 'max'),
        total_cases=('Cases', 'sum'),
        total_deaths=('Deaths', 'sum'),
        nonzero_case_records=('Cases', lambda x: (x > 0).sum()),
        zero_case_records=('Cases', lambda x: (x == 0).sum())
    )
    .reset_index()
)

# Percentages
disease_metadata['nonzero_case_%'] = (
    disease_metadata['nonzero_case_records']
    / disease_metadata['rows']
    * 100
)

disease_metadata['zero_case_%'] = (
    disease_metadata['zero_case_records']
    / disease_metadata['rows']
    * 100
)

# Sort by number of records
disease_metadata = disease_metadata.sort_values(
    by='rows',
    ascending=False
)

# ============================================================
# SAVE DISEASE METADATA
# ============================================================

disease_output = (
    'E:/Projects/BTP/Data/Clean/INDIA/'
    'disease_level_metadata.csv'
)

os.makedirs(
    os.path.dirname(disease_output),
    exist_ok=True
)

disease_metadata.to_csv(
    disease_output,
    index=False
)

print("\n" + "=" * 70)
print("DISEASE LEVEL METADATA")
print("=" * 70)

print(
    disease_metadata.to_string(index=False)
)

print("\n" + "=" * 70)
print("DISEASE METADATA SAVED")
print("=" * 70)

print(disease_output)


DISEASE LEVEL METADATA
                         Disease  rows  states  districts  unique_dates first_date  last_date  total_cases  total_deaths  nonzero_case_records  zero_case_records  nonzero_case_%  zero_case_%
        Acute Diarrhoeal Disease  5126      35        684          2610 2009-06-02 2022-06-29     251456.0        1265.0                  5126                  0      100.000000          0.0
                          Dengue  1619      34        344          1117 2009-07-28 2022-06-10     238047.0         613.0                  1618                  0       99.938233          0.0
                     Chikungunya   731      21        195           626 2009-06-10 2022-06-12      53289.0          14.0                   731                  0      100.000000          0.0
                         Cholera   666      25        208           545 2009-05-05 2022-06-06     126495.0         461.0                   666                  0      100.000000          0.0
                     

In [4]:
# ============================================================
# ACUTE DIARRHOEAL DISEASE - STATE LEVEL METADATA
# ============================================================

# Filter only Acute Diarrhoeal Disease
acute_df = df[
    df['Disease'].str.strip().str.lower()
    == 'acute diarrhoeal disease'
].copy()

# State-level statistics
acute_state_metadata = (
    acute_df.groupby('state_ut')
    .agg(
        rows=('state_ut', 'size'),
        unique_districts=('district', 'nunique'),
        unique_dates=('Date', 'nunique'),
        first_date=('Date', 'min'),
        last_date=('Date', 'max'),
        total_cases=('Cases', 'sum'),
        total_deaths=('Deaths', 'sum'),
        nonzero_case_records=('Cases', lambda x: (x > 0).sum()),
        zero_case_records=('Cases', lambda x: (x == 0).sum())
    )
    .reset_index()
)

# Calculate percentages
acute_state_metadata['nonzero_case_%'] = (
    acute_state_metadata['nonzero_case_records']
    / acute_state_metadata['rows']
    * 100
)

acute_state_metadata['zero_case_%'] = (
    acute_state_metadata['zero_case_records']
    / acute_state_metadata['rows']
    * 100
)

# Sort by number of records
acute_state_metadata = acute_state_metadata.sort_values(
    by='rows',
    ascending=False
)

# ============================================================
# SAVE
# ============================================================

acute_output = (
    'E:/Projects/BTP/Data/Clean/INDIA/'
    'acute_diarrhoeal_state_metadata.csv'
)

os.makedirs(
    os.path.dirname(acute_output),
    exist_ok=True
)

acute_state_metadata.to_csv(
    acute_output,
    index=False
)

# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 70)
print("ACUTE DIARRHOEAL DISEASE - STATE LEVEL METADATA")
print("=" * 70)

print(
    acute_state_metadata.to_string(index=False)
)

print("\n" + "=" * 70)
print("SAVED SUCCESSFULLY")
print("=" * 70)

print(f"File: {acute_output}")


ACUTE DIARRHOEAL DISEASE - STATE LEVEL METADATA
                   state_ut  rows  unique_districts  unique_dates first_date  last_date  total_cases  total_deaths  nonzero_case_records  zero_case_records  nonzero_case_%  zero_case_%
                West Bengal   562                37           421 2009-06-14 2022-04-25      42317.0          67.0                   562                  0           100.0          0.0
                  Karnataka   515                50           453 2009-06-09 2022-06-14      18019.0          61.0                   515                  0           100.0          0.0
                Maharashtra   445                35           397 2009-07-23 2022-06-15      23685.0          45.0                   445                  0           100.0          0.0
             Madhya Pradesh   371                55           321 2009-06-16 2022-06-12      17748.0         134.0                   371                  0           100.0          0.0
                    Gujara

In [6]:
# ============================================================
# ACUTE DIARRHOEAL DISEASE - WEEKLY COVERAGE METADATA
# ============================================================

acute_df = df[
    df['Disease'].str.strip().str.lower()
    == 'acute diarrhoeal disease'
].copy()

# Create proper date
acute_df['Date'] = pd.to_datetime(
    acute_df[['year', 'mon', 'day']].rename(columns={'mon': 'month'}),
    errors='coerce'
)

# Convert every date to the Monday of its week
acute_df['Week'] = (
    acute_df['Date']
    - pd.to_timedelta(acute_df['Date'].dt.weekday, unit='D')
)

# Aggregate to State × Week
weekly_state = (
    acute_df
    .groupby(['state_ut', 'Week'])
    .agg(
        total_cases=('Cases', 'sum'),
        total_deaths=('Deaths', 'sum'),
        reporting_districts=('district', 'nunique'),
        records_in_week=('district', 'size')
    )
    .reset_index()
)

# Calculate state-level weekly coverage
state_week_metadata = (
    weekly_state
    .groupby('state_ut')
    .agg(
        unique_weeks=('Week', 'nunique'),
        first_week=('Week', 'min'),
        last_week=('Week', 'max'),
        total_cases=('total_cases', 'sum'),
        total_deaths=('total_deaths', 'sum'),
        average_cases_per_week=('total_cases', 'mean'),
        max_cases_in_week=('total_cases', 'max'),
        average_districts_reporting=('reporting_districts', 'mean'),
        max_districts_reporting=('reporting_districts', 'max')
    )
    .reset_index()
)

# Calculate theoretical number of weeks between first and last week
state_week_metadata['calendar_weeks'] = (
    (
        state_week_metadata['last_week']
        - state_week_metadata['first_week']
    ).dt.days // 7
) + 1

# Percentage of weeks that actually contain reported data
state_week_metadata['weekly_coverage_%'] = (
    state_week_metadata['unique_weeks']
    / state_week_metadata['calendar_weeks']
    * 100
)

# Sort by number of unique weeks
state_week_metadata = state_week_metadata.sort_values(
    by='unique_weeks',
    ascending=False
)

# ============================================================
# SAVE CSV
# ============================================================

weekly_output = (
    'E:/Projects/BTP/Data/Clean/INDIA/'
    'acute_diarrhoeal_weekly_coverage_metadata.csv'
)

import os
os.makedirs(os.path.dirname(weekly_output), exist_ok=True)

state_week_metadata.to_csv(
    weekly_output,
    index=False
)

# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("ACUTE DIARRHOEAL DISEASE - WEEKLY COVERAGE")
print("=" * 80)

print(
    state_week_metadata.to_string(index=False)
)

print("\n" + "=" * 80)
print("CSV SAVED SUCCESSFULLY")
print("=" * 80)

print(weekly_output)


ACUTE DIARRHOEAL DISEASE - WEEKLY COVERAGE
                   state_ut  unique_weeks first_week  last_week  total_cases  total_deaths  average_cases_per_week  max_cases_in_week  average_districts_reporting  max_districts_reporting  calendar_weeks  weekly_coverage_%
                  Karnataka           283 2009-06-08 2022-06-13      18019.0          61.0               63.671378              454.0                     1.628975                        5             680          41.617647
                Maharashtra           265 2009-07-20 2022-06-13      23685.0          45.0               89.377358              834.0                     1.581132                        7             674          39.317507
                West Bengal           254 2009-06-08 2022-04-25      42317.0          67.0              166.602362             4597.0                     1.732283                        7             673          37.741456
                    Gujarat           233 2009-06-15 2022-06-27 